<h1>part 1: set up the whole system</h1>

In [26]:
!apt-get update
!apt-get install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

Get:1 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,899 kB]
Get:5 https://cli.github.com/packages stable/main amd64 Packages [356 B]       
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]           
Get:7 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Hit:8 http://archive.ubuntu.com/ubuntu jammy InRelease                         
Get:9 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [105 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]       
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,158 kB]
Get:12 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.

In [33]:
import os
import subprocess
import time
import requests
import subprocess

os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)
print(requests.get("http://127.0.0.1:11434/api/tags").json())

models = [
    "qwen2.5-coder:7b",
    "qwen2.5-coder:14b",
    # "qwen2.5-coder:32b",
    # "llama3.1:20b"  # Add any other 20B/32B models here
]

# Set of spinner characters and noise terms to completely filter out
NOISE_CHARS = set("⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏▕▏")
NOISE_TERMS = ["%", "MB/", "GB/", "verifying sha256"]

for model in models:
    print(f"\n🚀 Starting download: {model}...")
    
    process = subprocess.Popen(
        ["ollama", "pull", model],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True
    )
    
    last_status = ""
    for line in process.stdout:
        line_clean = line.strip()
        
        if not line_clean or line_clean == last_status:
            continue
            
        # Check if the line contains any of the noise patterns
        has_noise = (
            any(char in line_clean for char in NOISE_CHARS) or 
            any(term in line_clean for term in NOISE_TERMS)
        )
        
        if not has_noise:
            print(f"  [{model}]: {line_clean}")
            last_status = line_clean
                
    process.wait()
    if process.returncode == 0:
        print(f"✅ {model} downloaded successfully!")
    else:
        print(f"❌ Failed to download {model} (Exit code: {process.returncode})")

print("\n🎉 All requested models processed!")

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

time.sleep(5)
print(requests.get("http://127.0.0.1:11434/api/tags").json())

{'models': []}

🚀 Starting download: qwen2.5-coder:7b...
  [qwen2.5-coder:7b]: writing manifest 
  [qwen2.5-coder:7b]: success 
✅ qwen2.5-coder:7b downloaded successfully!

🚀 Starting download: qwen2.5-coder:14b...
  [qwen2.5-coder:14b]: writing manifest 
  [qwen2.5-coder:14b]: success 
✅ qwen2.5-coder:14b downloaded successfully!

🎉 All requested models processed!
{'models': [{'name': 'qwen2.5-coder:14b', 'model': 'qwen2.5-coder:14b', 'modified_at': '2026-08-08T09:58:23.086753572Z', 'size': 8988124298, 'digest': '9ec8897f747e246e970bc5cfdda85d22f1123dc2e3d34978a010a75968716849', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'qwen2', 'families': ['qwen2'], 'parameter_size': '14.8B', 'quantization_level': 'Q4_K_M', 'context_length': 32768, 'embedding_length': 5120}, 'capabilities': ['completion', 'tools', 'insert']}, {'name': 'qwen2.5-coder:7b', 'model': 'qwen2.5-coder:7b', 'modified_at': '2026-08-08T09:57:25.991926949Z', 'size': 4683087561, 'digest': 'dae161e27b0e90dd1856

In [34]:
!node -v
!npm install jsdom@22.1.0 --no-audit --no-fund

v20.19.0
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹npm warn deprecated whatwg-encoding@2.0.0: Use @exodus/bytes instead for a more spec-conformant and faster implementation
⠹npm warn deprecated abab@2.0.6: Use your platform's native atob() and btoa() methods instead
⠹npm warn deprecated domexception@4.0.0: Use your platform's native DOMException instead
⠹⠸⠼⠴⠦
added 58 packages in 3s
⠦npm notice
npm notice New major version of npm available! 10.8.2 -> 12.0.2
npm notice Changelog: https://github.com/npm/cli/releases/tag/v12.0.2
npm notice To update run: npm install -g npm@12.0.2
npm notice
⠦

<h1>part 2: fast extraction</h1>

In [41]:
from typing import List
from bs4 import BeautifulSoup, Tag
import os
import re
import json
import difflib
from bs4 import BeautifulSoup
from typing import Tuple
import json
from pathlib import Path
from tqdm.auto import tqdm
import traceback

MIN_HTML_LENGTH = 150
MAX_HTML_LENGTH = 100000
PERSIAN_ARABIC_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")


def save_final_layers(final_layers, filename="final_layers.json", folder_name="processed_layers"):
    output_dir = Path("/kaggle/working") / folder_name
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / filename

    with output_path.open("w", encoding="utf-8") as file:
        json.dump(
            final_layers,
            file,
            ensure_ascii=False,
            indent=2
        )

    return str(output_path)


def extract_dom_layers(html: str) -> List[str]:
    """
    دقیقاً یک لایه به داخل قطعه HTML ورودی می‌رود و تمام فرزندان مستقیمی
    که خودشان والد هستند (حداقل یک تگ فرزند دارند) را همراه با تمام زیرمجموعه‌شان
    به عنوان رشته‌های HTML مجزا برمی‌گرداند.
    """
    if not isinstance(html, str):
        raise TypeError("html باید از نوع str باشد.")

    soup = BeautifulSoup(html, "html.parser")
    top_level_tags = [node for node in soup.contents if isinstance(node, Tag)]

    if not top_level_tags:
        return []

    parents_html = []

    for root_tag in top_level_tags:
        for child in root_tag.children:
            if isinstance(child, Tag):
                # بررسی اینکه آیا این فرزند خودش والد است (حداقل یک فرزند تگی دارد)
                has_child_tag = any(isinstance(c, Tag) for c in child.children)
                if has_child_tag:
                    parents_html.append(str(child))

    return parents_html



def clean_text_and_normalize(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")
    text = soup.get_text(separator=" ")
    normalized_text = text.translate(PERSIAN_ARABIC_DIGITS)
    normalized_text = re.sub(r'[,\u066C،٬]', '', normalized_text)
    return normalized_text


def price_validation(html: str) -> bool:
    text = clean_text_and_normalize(html)
    numbers = re.findall(r'\d+', text)

    for num_str in numbers:
        try:
            val = int(num_str)
            if 1_000_000 <= val <= 1_000_000_000:
                return True
        except ValueError:
            continue

    return False

def kill_process(html: str) -> bool:
    text = clean_text_and_normalize(html)
    numbers = re.findall(r'\d+', text)

    counter = 0
    for num_str in numbers:
        try:
            val = int(num_str)
            if 1_000_000 <= val <= 1_000_000_000:
                counter += 1
        except ValueError:
            continue

    return counter <= 4



def time_validation(html: str) -> bool:
    normalized_html = html.translate(PERSIAN_ARABIC_DIGITS)
    time_pattern = r'\b(?:[0-1]?[0-9]|2[0-3]):[0-5][0-9]\b'
    if re.search(time_pattern, normalized_html):
        return True
    return False


def hot_validate_html(html: str):
    html_len = len(html)
    length_valid = MIN_HTML_LENGTH <= html_len <= MAX_HTML_LENGTH

    price = price_validation(html)
    if not price:
        return {"price": False,"time": "", "length": length_valid}

    time = time_validation(html)
    if not time:
        return {"price": True, "time": False, "length": length_valid}

    return {"price": True,"time": True, "length": length_valid}

def fast_safe_extraction(
    file_path,
    number,
    output_prefix="final_layers",
    folder_name="processed_layers"
):
    """
    یک فایل HTML را پردازش می‌کند و لایه‌های احتمالی کارت پرواز
    را در یک پوشه اختصاصی ذخیره می‌کند (بدون نمایش پروسه داخلی).
    """
    file_path = Path(file_path)

    if not file_path.exists():
        raise FileNotFoundError(f"فایل در مسیر مشخص‌شده یافت نشد: {file_path}")

    if not file_path.is_file():
        raise ValueError(f"مسیر داده‌شده یک فایل نیست: {file_path}")

    # خواندن محتوای HTML
    html = file_path.read_text(encoding="utf-8")

    # استخراج اولین سطح DOM
    current_layers = extract_dom_layers(html)

    final_layers = []
    layer_number = 1
    total_checked = 0

    while current_layers:
        next_layers = []

        for chunk in current_layers:
            total_checked += 1

            if not isinstance(chunk, str) or not chunk.strip():
                continue

            check = hot_validate_html(chunk)

            has_price = check["price"] is True
            has_time = check["time"] is True
            valid_length = check["length"] is True

            # بررسی و فیلتر کارت‌ها
            if has_price and has_time and valid_length:
                if kill_process(chunk):
                    final_layers.append(chunk)
                else:
                    children = extract_dom_layers(chunk)
                    if children:
                        next_layers.extend(children)
            else:
                # بررسی فرزندان حتی اگر والد مستقیم فاقد ساختار کامل کارت باشد
                children = extract_dom_layers(chunk)
                if children:
                    next_layers.extend(children)

        if not next_layers:
            break

        current_layers = next_layers
        layer_number += 1

    # حذف HTMLهای تکراری
    unique_final_layers = list(dict.fromkeys(final_layers))
    
    # نام‌گذاری خروجی
    output_filename = f"{output_prefix}_{number:03d}.json"

    # ذخیره در پوشه اختصاصی
    output_path = save_final_layers(
        unique_final_layers,
        filename=output_filename,
        folder_name=folder_name
    )

    print(
        f"✅ File {number}: {file_path.name} | "
        f"Found: {len(unique_final_layers)} | "
        f"Saved to: {folder_name}/{output_filename}"
    )

    return {
        "input_file": str(file_path),
        "input_name": file_path.name,
        "output_file": output_path,
        "tickets_found": len(unique_final_layers),
        "chunks_checked": total_checked,
        "layers_processed": layer_number
    }



INPUT_DIR = Path("/kaggle/input/datasets/rezapourmoridi/ticketsalespages")
FOLDER_NAME = "processed_layers"
if not INPUT_DIR.exists():
    raise FileNotFoundError(f"مسیر دیتاست یافت نشد: {INPUT_DIR}")
html_files = sorted(
    [
        path
        for path in INPUT_DIR.rglob("*")
        if path.is_file() and path.suffix.lower() in {".html", ".htm"}
    ]
)

if not html_files:
    raise FileNotFoundError(f"هیچ فایل HTML در مسیر یافت نشد.")

print(f"🔎 Found {len(html_files)} HTML files to process.")
print(f"📁 Saving outputs to: /kaggle/working/{FOLDER_NAME}/\n")

results = []
failed_files = []

for number, file_path in enumerate(
    tqdm(html_files, desc="Batch Processing HTMLs", unit="file"),
    start=1
):
    try:
        result = fast_safe_extraction(
            file_path=file_path,
            number=number,
            output_prefix="final_layers",
            folder_name=FOLDER_NAME,
        )
        results.append(result)
    except Exception as error:
        failed_files.append({
            "number": number,
            "file": str(file_path),
            "error": repr(error)
        })
        print(f"\n❌ Error on file {number} ({file_path.name}): {error}")

summary_path = Path("/kaggle/working") / FOLDER_NAME / "batch_summary.json"
summary = {
    "total_files": len(html_files),
    "successful_files": len(results),
    "failed_files": len(failed_files),
    "results": results,
    "failures": failed_files
}

with summary_path.open("w", encoding="utf-8") as file:
    json.dump(summary, file, ensure_ascii=False, indent=2)

print("\n" + "=" * 50)
print("🏁 Extraction Completed!")
print(f"Processed: {len(results)} | Failed: {len(failed_files)}")
print(f"Summary log created at: {summary_path}")
print("=" * 50)


🔎 Found 22 HTML files to process.
📁 Saving outputs to: /kaggle/working/processed_layers/



Batch Processing HTMLs:   0%|          | 0/22 [00:00<?, ?file/s]

✅ File 1: 20260807_144935_www_alibaba_ir_flights_THR-MHD.html | Found: 7 | Saved to: processed_layers/final_layers_001.json
✅ File 2: 20260807_145045_www_alibaba_ir_flights_THR-MHD.html | Found: 1 | Saved to: processed_layers/final_layers_002.json
✅ File 3: 20260807_145143_www_alibaba_ir_international_IKA-ISTALL.html | Found: 18 | Saved to: processed_layers/final_layers_003.json
✅ File 4: 20260807_145252_www_alibaba_ir_international_IKA-ISTALL.html | Found: 18 | Saved to: processed_layers/final_layers_004.json
✅ File 5: 20260807_145548_www_snapptrip_ir_flights_THR_city_MHD_city.html | Found: 11 | Saved to: processed_layers/final_layers_005.json
✅ File 6: 20260807_145646_www_snapptrip_ir_flights_THR_city_MHD_city.html | Found: 10 | Saved to: processed_layers/final_layers_006.json
✅ File 7: 20260807_145747_www_snapptrip_ir_inter-flights_THR_city_IST_city.html | Found: 27 | Saved to: processed_layers/final_layers_007.json
✅ File 8: 20260807_145854_www_snapptrip_ir_inter-flights_THR_city_D

<h1>part 3: final extraction</h1>

In [43]:
import json
import os
from typing import Tuple
import math
import subprocess
import tempfile
import requests
import subprocess
import time
from tqdm import tqdm # برای نمایش نوار پیشرفت
import re
from pathlib import Path
from typing import List, Tuple
from bs4 import BeautifulSoup, Tag


OLLAMA_BASE_URL = os.getenv("OLLAMA_HOST", "http://127.0.0.1:11434").rstrip("/")
OLLAMA_API_URL = f"{OLLAMA_BASE_URL}/api/generate"
# MODEL_NAME = os.getenv("OLLAMA_MODEL", "qwen2.5-coder:7b")
MODEL_NAME = os.getenv("OLLAMA_MODEL", "qwen2.5-coder:14b")
REQUEST_TIMEOUT = int(os.getenv("OLLAMA_TIMEOUT", "360"))
MAX_HTML_CHARS = int(os.getenv("FINAL_VALIDATION_MAX_HTML_CHARS", "100000"))

NODE_WORKDIR = os.getcwd()
NODE_PATH = os.path.join(NODE_WORKDIR, "node_modules")
OLLAMA_PROCESS = None
PERSIAN_ARABIC_DIGITS = str.maketrans(
    "۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩",
    "01234567890123456789"
)

def clean_text_and_normalize(html: str) -> str:
    soup = BeautifulSoup(html, "html.parser")

    text = soup.get_text(separator=" ")
    text = text.translate(PERSIAN_ARABIC_DIGITS)
    text = re.sub(r"[,\u066C،٬]", "", text)
    text = re.sub(r"\s+", " ", text)

    return text.strip()

def extract_dom_layers(html: str) -> List[str]:
    """
    دقیقاً یک لایه به داخل قطعه HTML ورودی می‌رود و تمام فرزندان مستقیمی
    که خودشان والد هستند (حداقل یک تگ فرزند دارند) را همراه با تمام زیرمجموعه‌شان
    به عنوان رشته‌های HTML مجزا برمی‌گرداند.
    """
    if not isinstance(html, str):
        raise TypeError("html باید از نوع str باشد.")

    soup = BeautifulSoup(html, "html.parser")
    top_level_tags = [node for node in soup.contents if isinstance(node, Tag)]

    if not top_level_tags:
        return []

    parents_html = []

    for root_tag in top_level_tags:
        for child in root_tag.children:
            if isinstance(child, Tag):
                # بررسی اینکه آیا این فرزند خودش والد است (حداقل یک فرزند تگی دارد)
                has_child_tag = any(isinstance(c, Tag) for c in child.children)
                if has_child_tag:
                    parents_html.append(str(child))

    return parents_html


def start_ollama_server():
    global OLLAMA_PROCESS

    if OLLAMA_PROCESS is not None and OLLAMA_PROCESS.poll() is None:
        return True

    OLLAMA_PROCESS = subprocess.Popen(
        ["ollama", "serve"],
        stdout=open("/kaggle/working/ollama_stdout.log", "a"),
        stderr=open("/kaggle/working/ollama_stderr.log", "a"),
    )

    for _ in range(20):
        try:
            r = requests.get(f"{OLLAMA_BASE_URL}/api/tags", timeout=2)
            if r.status_code == 200:
                return True
        except requests.RequestException:
            pass
        time.sleep(1)

    return False


def clean_html_with_js(html: str) -> str:
    # Safety: cap raw input BEFORE jsdom parses it.
    # jsdom can be memory-heavy on very large/pathological HTML.
    if len(html) > MAX_HTML_CHARS:
        html = html[:MAX_HTML_CHARS]

    js_code = r"""

const fs = require("fs");
const { JSDOM, VirtualConsole } = require("jsdom");

const inputPath = process.argv[2];
const html = fs.readFileSync(inputPath, "utf8");

const virtualConsole = new VirtualConsole();

const dom = new JSDOM(html, {
    virtualConsole
});

const document = dom.window.document;
const NodeFilter = dom.window.NodeFilter;

const clone = document.documentElement.cloneNode(true);

const REMOVE_TAGS = new Set([
    "svg",
    "symbol",
    "use"
]);

function isHiddenElement(el) {
    const style = (el.getAttribute("style") || "")
        .toLowerCase()
        .replace(/\s+/g, "");

    return (
        el.hasAttribute("hidden") ||
        el.getAttribute("aria-hidden") === "true" ||
        style.includes("display:none") ||
        style.includes("visibility:hidden")
    );
}

// Remove HTML comments
{
    const walker = document.createTreeWalker(
        clone,
        NodeFilter.SHOW_COMMENT
    );

    const comments = [];
    while (walker.nextNode()) {
        comments.push(walker.currentNode);
    }

    for (const node of comments) {
        node.remove();
    }
}

// Remove only SVGs and hidden elements
{
    const elements = Array.from(clone.querySelectorAll("*")).reverse();

    for (const el of elements) {
        const tag = el.tagName.toLowerCase();

        if (REMOVE_TAGS.has(tag)) {
            el.remove();
            continue;
        }

        if (isHiddenElement(el)) {
            el.remove();
        }
    }
}

process.stdout.write("<!DOCTYPE html>\n" + clone.outerHTML);


"""

    with tempfile.NamedTemporaryFile("w", suffix=".html", encoding="utf-8", delete=False) as html_file:
        html_file.write(html)
        html_path = html_file.name

    with tempfile.NamedTemporaryFile("w", suffix=".js", encoding="utf-8", delete=False) as js_file:
        js_file.write(js_code)
        js_path = js_file.name

    try:
        result = subprocess.run(
            [
                "node",
                js_path,
                html_path
            ],
            capture_output=True,
            text=True,
            check=True,
            cwd=NODE_WORKDIR,
            env={
                **os.environ,
                "NODE_PATH": NODE_PATH,
            },
        )

        return result.stdout

    except subprocess.CalledProcessError as exc:
        print("\nHTML cleaner failed, using original HTML")
        print("STDERR:")
        print(exc.stderr)
        return html

    except subprocess.TimeoutExpired as exc:
        print(f"\nHTML cleaner timeout, using original HTML: {exc}")
        return html

    except Exception as exc:
        print(f"\nHTML cleaner failed, using original HTML: {exc}")
        return html

    finally:
        try:
            os.remove(html_path)
        except OSError:
            pass

        try:
            os.remove(js_path)
        except OSError:
            pass

def final_kill_process(html: str) -> bool:
    text = clean_text_and_normalize(html)
    numbers = re.findall(r'\d+', text)

    counter = 0
    for num_str in numbers:
        try:
            val = int(num_str)
            if 1_000_000 <= val <= 1_000_000_000:
                counter += 1
        except ValueError:
            continue

    return counter < 1


def final_validation(html_chunk: str)  -> Tuple[bool, float]:

    html_chunk = clean_html_with_js(html_chunk)
    # print(html_chunk[:MAX_HTML_CHARS])
    system_rules = """
    You are a strict whole-chunk HTML binary classifier.
    
    Classify the complete provided HTML chunk, not merely whether some text inside it
    looks like a flight ticket.
    
    Return exactly one JSON object with exactly these keys:
    {
      "is_ticket": boolean,
      "confidence": number,
      "why": string
    }
    
    The result must be false if the HTML is a parent section, wrapper, list,
    search-results container, airline filter, or collection containing one or more
    flight-related options.
    
    Never infer is_ticket=true merely because the chunk contains flight names,
    airlines, times, prices, or a booking button.
    
    Do not output markdown, explanations outside the JSON, or extra keys.
    """

        
    prompt = f"""
    Classify whether the following HTML chunk represents exactly ONE complete standalone actionable flight ticket card.
    
    <rules>
    Return true ONLY if the chunk contains:
    - one airline section
    - route/origin and destination
    - departure or arrival time
    - price
    - actionable/selectable CTA
    
    Important:
    You do NOT have reliable knowledge of all airline names.
    Do NOT reject a card just because the airline name is unfamiliar.
    Infer whether an airline is present based on layout, structure, positioning, logos, labels, and surrounding flight-related context.
    
    Return false if ANY apply:
    - Contains multiple ticket cards
    - Contains wrapper/list/container elements
    - Contains banners, ads, hotels, headers, filters, sticky bars, summaries, or unrelated content
    - Is only a partial component
    - One of its child elements is itself a complete valid ticket card
    - Route, timing, airline section, or price is unclear or missing
    - if it contains a ticket card plus any unrelated sibling or external UI section, including filters, result-list controls, headers, banners, ads, pagination, or additional content.

    
    Responsive mobile/desktop duplicate elements inside the same card do NOT count as multiple cards.
    </rules>
    
    HTML:
    \"\"\"{html_chunk[:MAX_HTML_CHARS]}\"\"\"
    CRITICAL Reminder:
    Return true ONLY if the chunk contains:airline, route-destination, time, price.
    Return ONLY valid JSON.
    Return false for multiple tickets only when the chunk contains at least two distinct itinerary offers. Multiple nested elements, grid columns, labels, or visual sections do not by themselves indicate multiple cards.
    Return false if it contains a ticket card plus any unrelated sibling or external UI section, including filters, result-list controls, headers, banners, ads, pagination, or additional content.
    
    Schema:
    {{"is_ticket": <boolean>, "confidence": <float>, "why": "<very short reason>"}}
    """



    estimated_tokens = math.ceil(len(prompt) / 2) + 150
    dynamic_num_ctx = math.ceil(estimated_tokens*2.6 / 64) * 64
    dynamic_num_ctx = max(8192, min(dynamic_num_ctx, 64000))
    print(dynamic_num_ctx)

    payload = {
        "model": MODEL_NAME,
        "system": system_rules,
        "prompt": prompt,
        "stream": False,
        "format": "json",
        # "keep_alive": "30m",
        "options": {
            "temperature": 0.15,
            "num_predict": 440,
            "num_ctx": dynamic_num_ctx
        }
    }

    try:
        response = requests.post(
            OLLAMA_API_URL,
            json=payload,
            timeout=(5, REQUEST_TIMEOUT),
        )
        response.raise_for_status()

    except requests.exceptions.Timeout as exc:
        print(f"\nOllama timeout: {exc}")
        return False, 0.0

    except requests.exceptions.RequestException as exc:
        print(f"\nOllama request failed: {exc}")
        return False, 0.0

    try:
        result_json = response.json()
        response_text = result_json.get("response", "").strip()
        data = json.loads(response_text)
        print(data)

    except (ValueError, json.JSONDecodeError, TypeError) as exc:
        print(f"\nInvalid Ollama response: {exc}")
        return False, 0.0

    is_ticket = bool(data.get("is_ticket", False))

    try:
        confidence = float(data.get("confidence", 0.0))
    except (TypeError, ValueError):
        confidence = 0.0

    confidence = max(0.0, min(1.0, confidence))
    return is_ticket, confidence

def save_validated_tickets(
    tickets,
    output_dir,
    output_filename,
):
    """
    ذخیره بلیت‌های تأییدشده در پوشه خروجی.
    """

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_path = output_dir / output_filename

    with output_path.open("w", encoding="utf-8") as f:
        json.dump(
            tickets,
            f,
            ensure_ascii=False,
            indent=2,
        )

    return output_path




def process_file_validation(
    file_path,
    file_number,
    output_dir="/kaggle/working/validated_tickets",
):
    """
    یک فایل final_layers را پردازش می‌کند و خروجی متناظر آن را می‌سازد.
    """

    file_path = Path(file_path)

    with file_path.open("r", encoding="utf-8") as f:
        current_layers = json.load(f)

    if not isinstance(current_layers, list):
        raise ValueError(
            f"ساختار فایل {file_path.name} باید یک لیست JSON باشد."
        )

    validated_tickets = []
    layer_count = 1

    while current_layers:
        next_layers = []

        progress_description = (
            f"{file_path.stem} | layer {layer_count}"
        )

        for chunk in tqdm(
            current_layers,
            desc=progress_description,
            unit="chunk",
            leave=False,
        ):
            if not isinstance(chunk, str):
                continue
            if final_kill_process(chunk):
                continue
            try:
                is_ticket, confidence = final_validation(chunk)
            except Exception:
                is_ticket, confidence = False, 0.0

            if is_ticket:
                validated_tickets.append(chunk)
            else:
                children = extract_dom_layers(chunk)
                if children:
                    next_layers.extend(children)
        if not next_layers:
            break

        current_layers = next_layers
        layer_count += 1

    # حذف duplicateهای احتمالی، بدون تغییر ترتیب
    validated_tickets = list(dict.fromkeys(validated_tickets))

    output_filename = (
        f"validated_tickets_{file_number:03d}.json"
    )

    output_path = save_validated_tickets(
        tickets=validated_tickets,
        output_dir=output_dir,
        output_filename=output_filename,
    )

    return {
        "input_file": file_path.name,
        "output_file": output_path.name,
        "tickets": len(validated_tickets),
        "output_path": str(output_path),
    }


def run_pipeline(
    input_dir="/kaggle/working/processed_layers",
    output_dir="/kaggle/working/validated_tickets",
):
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.exists():
        raise FileNotFoundError(
            f"پوشه ورودی پیدا نشد: {input_dir}"
        )

    layer_files = sorted(
        input_dir.glob("final_layers_*.json")
    )

    if not layer_files:
        raise FileNotFoundError(
            f"هیچ فایل final_layers_*.json در {input_dir} پیدا نشد."
        )

    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"Files found: {len(layer_files)}")

    if not start_ollama_server():
        raise RuntimeError(
            "Ollama server is not available."
        )

    results = []
    total_tickets = 0

    for file_number, file_path in enumerate(
        tqdm(
            layer_files,
            desc="Overall progress",
            unit="file",
        ),
        start=1,
    ):
        try:
            result = process_file_validation(
                file_path=file_path,
                file_number=file_number,
                output_dir=output_dir,
            )

            results.append(result)
            total_tickets += result["tickets"]

        except Exception as exc:
            results.append({
                "input_file": file_path.name,
                "output_file": None,
                "tickets": 0,
                "error": str(exc),
            })

    successful_files = sum(
        1 for item in results
        if "error" not in item
    )

    failed_files = len(results) - successful_files

    print("\n" + "=" * 50)
    print("Pipeline finished")
    print(f"Processed files: {successful_files}")
    print(f"Failed files: {failed_files}")
    print(f"Validated tickets: {total_tickets}")
    print(f"Output directory: {output_dir}")
    print("=" * 50)

    return results

results = run_pipeline()

Files found: 22


Overall progress:   0%|          | 0/22 [00:00<?, ?file/s]
final_layers_001 | layer 1:   0%|          | 0/7 [00:00<?, ?chunk/s]

18752



final_layers_001 | layer 1:  14%|█▍        | 1/7 [00:14<01:28, 14.77s/chunk]

{'is_ticket': False, 'confidence': 0.99, 'why': 'The HTML chunk is a sidebar with filters and options, not a standalone flight ticket card.'}
8192


...

{'is_ticket': True, 'confidence': 0.95, 'why': 'Contains airline logo, route/destination, time, price, and CTA button.'}
9088



final_layers_019 | layer 1:  29%|██▉       | 82/278 [08:56<21:57,  6.72s/chunk]

{'is_ticket': True, 'confidence': 0.95, 'why': 'Contains airline logo, route/destination, time, price, and CTA button.'}
9088



Overall progress:  82%|████████▏ | 18/22 [1:42:43<22:49, 342.40s/file]         


KeyboardInterrupt: 

<h1>part 4: get itmes</h1>

In [32]:
import json
from bs4 import BeautifulSoup, Tag
threshold = 0.95

def normalize_tag(tag: Tag):
    classes = tuple(tag.get("class", []))

    children = tuple(
        normalize_tag(child)
        for child in tag.children
        if isinstance(child, Tag)
    )
    return tag.name, classes, children


def get_root_tag(html: str) -> Tag:
    soup = BeautifulSoup(html, "html.parser")
    root_tags = [
        tag for tag in soup.contents
        if isinstance(tag, Tag)
    ]
    if len(root_tags) != 1:
        raise ValueError("هر item باید دقیقاً یک تگ ریشه داشته باشد.")
    return root_tags[0]

# add this
def tag_similarity(a: Tag, b: Tag) -> float:
    score = 0.0
    total = 0.0

    # tag name
    total += 1
    score += 1.0 if a.name == b.name else 0.0

    # classes
    total += 1
    a_classes = set(a.get("class", []))
    b_classes = set(b.get("class", []))
    if a_classes or b_classes:
        score += len(a_classes & b_classes) / len(a_classes | b_classes)
    else:
        score += 1.0

    # children
    total += 1
    a_children = [c for c in a.children if isinstance(c, Tag)]
    b_children = [c for c in b.children if isinstance(c, Tag)]

    if not a_children and not b_children:
        score += 1.0
    elif a_children and b_children:
        n = min(len(a_children), len(b_children))
        child_score = sum(
            tag_similarity(a_children[i], b_children[i]) for i in range(n)
        ) / max(len(a_children), len(b_children))
        score += child_score
    else:
        score += 0.0

    return score / total


def find_all_matching_items(page_html: str, items: list[str]):
    page_soup = BeautifulSoup(page_html, "html.parser")
    page_tags = page_soup.find_all(True)

    item_roots = [
        get_root_tag(item)
        for item in items
    ]
    unique_matches = []
    
    for page_tag in page_tags:
        if any(
            tag_similarity(page_tag, item_root) >= threshold
            for item_root in item_roots
        ):
            unique_matches.append(page_tag)
    
    return list(dict.fromkeys(unique_matches))



with open(
    "/kaggle/input/datasets/rezapourmoridi/ticketsalespages/20260807_151041_ghasedak24_com_flights_THR-MHD.html",
    "r",
    encoding="utf-8"
) as f:
    html_content = f.read()

with open(
    "/kaggle/input/datasets/rezapourmoridi/ticket-itmes/validated_tickets_017.json",
    "r",
    encoding="utf-8"
) as f:
    items = json.load(f)

print(len(items))

matches = find_all_matching_items(
    page_html=html_content,
    items=items
)
print(f"\nTotal unique matches: {len(matches)}")
for item in matches:
    print(item)
    print(' ---- ')

13

Total unique matches: 4
<div class="ghk-relative ghk-bg-white ghk-rounded-lg ghk-border-solid ghk-border ghk-border-gray-300 ghk-mb-4"><!-- --><div class="ghk-cursor-pointer ghk-hidden md:ghk-block"><img alt="جزییات" class="ghk-absolute ghk-bottom-0 ghk-right-1/4" src="data:image/svg+xml,%3csvg%20width='220'%20height='32'%20viewBox='0%200%20220%2032'%20fill='none'%20xmlns='http://www.w3.org/2000/svg'%3e%3cpath%20fill-rule='evenodd'%20clip-rule='evenodd'%20d='M43.0506%2016C28.8065%2024%2014.5625%2032%20-0.00146484%2032H219.207C204.643%2032%20190.399%2024%20176.155%2016C161.911%208%20147.667%200%20133.103%200H86.1026C71.5386%200%2057.2946%208%2043.0506%2016Z'%20fill='%23F9FAFB'/%3e%3c/svg%3e"/><div class="ghk-absolute ghk-bottom-1 ghk-right-1/4 ghk-pr-[78px] ghk-flex"><span class="ghk-text-xs16 ghk-pl-2 ghk-text-blue-primary"> جزییات </span><img alt="chevron" class="ghk-scale-75 ghk-rotate-90" src="data:image/svg+xml,%3csvg%20width='9'%20height='19'%20viewBox='0%200%209%2019'%20fill=

In [48]:
import os
import json
import re
from bs4 import BeautifulSoup
from difflib import get_close_matches

airlines = {
  "ایران ایر": ["ایران ایر", "هواپیمایی جمهوری اسلامی ایران", "هما", "Islamic Republic of Iran Airlines", "Homa", "Iran Air", "IranAir"],
  "ماهان": ["ماهان", "هواپیمایی ماهان", "ماهان ایر", "Mahan Air", "Mahan"],
  "آسمان": ["آسمان", "هواپیمایی آسمان", "ایران آسمان", "Aseman Airlines", "Iran Aseman Airlines", "Aseman"],
  "آتا": ["آتا", "هواپیمایی آتا", "آتا ایرلاین", "Ata Airlines", "Ata Airline", "Ata"],
  "ایران ایرتور": ["ایران ایرتور", "هواپیمایی ایران ایرتور", "هواپیمایی ایران ایرتور چارتر", "Iran Airtour Airline", "Iran Airtour", "Charter Airline", "Airtour"],
  "کیش ایر": ["کیش ایر", "هواپیمایی کیش ایر", "Kish Air", "Kish"],
  "قشم ایر": ["قشم ایر", "هواپیمایی قشم ایر", "Qeshm Air", "Qeshm"],
  "کاسپین": ["کاسپین", "هواپیمایی کاسپین", "کاسپین ایرلاین", "Caspian Airlines", "Caspian"],
  "زاگرس": ["زاگرس", "هواپیمایی زاگرس", "زاگرس ایرلاین", "Zagros Airlines", "Zagros"],
  "زاگرس قشم": ["زاگرس قشم", "هواپیمایی زاگرس قشم", "Zagros Qeshm"],
  "تابان": ["تابان", "هواپیمایی تابان", "تابان ایر", "Taban Air", "Taban"],
  "سپهران": ["سپهران", "هواپیمایی سپهران", "سپهران ایرلاین", "Sepehran Airlines", "Sepehran"],
  "وارش": ["وارش", "هواپیمایی وارش", "وارش ایرلاین", "Varesh Airlines", "Varesh"],
  "کارون": ["کارون", "هواپیمایی کارون", "کارون ایرلاین", "Karun Airlines", "Karun", "هواپیمایی نفت", "نفت ایر", "نفت", "Naft Air", "Naft"],
  "چابهار": ["چابهار", "هواپیمایی چابهار", "چابهار ایرلاین", "Chabahar Airlines", "Chabahar"],
  "فلای پرشیا": ["فلای پرشیا", "هواپیمایی فلای پرشیا", "Fly Persia", "FlyPersia"],
  "آوا ایر": ["آوا ایر", "هواپیمایی آوا ایر", "Ava Air", "Ava"],
  "معراج": ["معراج", "هواپیمایی معراج", "معراج ایر", "Meraj Airlines", "Meraj"],
  "ساها": ["ساها", "هواپیمایی ساها", "ساها ایر", "Saha Airlines", "Saha"],
  "پویا": ["پویا", "هواپیمایی پویا", "پویا ایر", "Pouya Air", "Pouya"],
  "پارس ایر": ["پارس ایر", "هواپیمایی پارس ایر", "Pars Air", "Pars"],
  "یزد ایر": ["یزد ایر", "هواپیمایی یزد ایر", "Yazd Air", "Yazd"],
  "ایر وان": ["ایر وان", "هواپیمایی ایر وان", "Air One", "AirOne"],
  "آساجت": ["آساجت", "هواپیمایی آساجت", "Asa Jet", "AsaJet"],
  "رایمون": ["رایمون", "هواپیمایی رایمون", "رایمون ایر", "Raymon Air", "Raymon"],
  "مهر": ["مهر", "هواپیمایی مهر", "مهر ایر", "Mehr Airlines", "Mehr"],
  "اطلس": ["اطلس", "هواپیمایی اطلس", "اطلس ایر", "Atlas Air", "Atlas"],
  "فلای کیش": ["فلای کیش", "هواپیمایی فلای کیش", "Fly Kish"],
  "فجر": ["فجر", "هواپیمایی فجر", "فجر ایر", "Fajr Air", "Fajr"],
  "آریا": ["آریا", "هواپیمایی آریا", "آریا ایر", "Aria Air", "Aria"],
  "یاس": ["یاس", "هواپیمایی یاس", "یاس ایر", "Yas Air", "Yas"],
  "سورینت": ["سورینت", "هواپیمایی سورینت", "سورینت ایر", "Surinet Air", "Surinet"],
  "آرمان": ["آرمان", "هواپیمایی آرمان", "آرمان ایر", "Arman Air", "Arman"],
  "سیمرغ": ["سیمرغ", "هواپیمایی سیمرغ", "سیمرغ ایر", "Simorgh Air", "Simorgh"],
  "پارسیان": ["پارسیان", "هواپیمایی پارسیان", "پارسیان ایر", "Parsian Air", "Parsian"],
  "نسیم": ["نسیم", "هواپیمایی نسیم", "خطوط هواپیمایی نسیم", "Nasim Airlines", "Nasim"],
  "ایران ایر شارجه": ["ایران ایر شارجه", "هواپیمایی ایران ایر شارجه", "CPN"],
  "باری": ["باری", "هواپیمایی باری", "Cargo Airline"],
    "سروش": ["سروش", "soroush"],
  "اختصاصی/خصوصی": ["اختصاصی", "خصوصی", "هواپیمایی اختصاصی", "هواپیمایی خصوصی", "Private Airline"]
}

AIRLINE_ALIASES = sorted(
    ((alias, key) for key, aliases in airlines.items() for alias in aliases),
    key=lambda pair: len(pair[0]),
    reverse=True,
)

PERSIAN_ARABIC_DIGITS = str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩", "01234567890123456789")

def clean_text_and_normalize(html: str) -> str:
    text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)
    text = text.translate(PERSIAN_ARABIC_DIGITS)
    text = re.sub(r'[,\u066C،٬]', '', text)
    return text

def get_price(html: str) -> int:
    text = clean_text_and_normalize(html)
    for num_str in re.findall(r'\d+', text):
        val = int(num_str)
        if 1_000_000 <= val <= 1_000_000_000:
            return val
    return 0

def get_time(html: str):
    text = html.translate(PERSIAN_ARABIC_DIGITS)
    m = re.search(r'\b(?:[0-1]?\d|2[0-3]):[0-5]\d\b', text)
    return m.group(0) if m else ""

def get_full_text(html: str) -> str:
    return clean_text_and_normalize(html)

ALIAS_LOOKUP = {alias.lower(): key for key, aliases in airlines.items() for alias in aliases}
ALL_ALIASES_LOWER = list(ALIAS_LOOKUP.keys())

def get_airline(text: str, THRESHOLD=0.8) -> str:
    text_lower = text.lower()
    tokens = text_lower.split()
    
    # لیستی برای جمع‌آوری تمام احتمالات
    candidates = []

    def check_and_add(phrase):
        # چک کردن دقیق (Direct Match)
        if phrase in ALIAS_LOOKUP:
            candidates.append(ALIAS_LOOKUP[phrase])
            return True
        # چک کردن فازی (Fuzzy Match)
        matches = get_close_matches(phrase, ALL_ALIASES_LOWER, n=1, cutoff=THRESHOLD)
        if matches:
            candidates.append(ALIAS_LOOKUP[matches[0]])
            return True
        return False

    # ۱. جستجوی ۳ کلمه‌ای (اولویت اول - مثل "ایران ایر تور")
    for i in range(len(tokens) - 2):
        check_and_add(f"{tokens[i]} {tokens[i+1]} {tokens[i+2]}")

    # ۲. جستجوی ۲ کلمه‌ای (اولویت دوم - مثل "کیش ایر")
    for i in range(len(tokens) - 1):
        check_and_add(f"{tokens[i]} {tokens[i+1]}")

    # ۳. جستجوی تک‌کلمه‌ای (اولویت سوم - مثل "ماهان")
    for token in tokens:
        if len(token) >= 3:
            check_and_add(token)

    # انتخاب هوشمندانه:
    # اگر "ایران ایر تور" پیدا شده باشد، در لیست candidates ما [..., "ایران ایرتور", "ایران ایر"] داریم.
    # کافیست لیستی را برگردانیم که بیشترین طول کاراکتری را دارد (چون خاص‌تر است).
    if candidates:
        # برگرداندن آیتمی که نام آن در دیتای اصلی طولانی‌تر است (دقت بالاتر)
        return max(candidates, key=len)
            
    return ""



def extract_ticket_item(item):
    html = item if isinstance(item, str) else json.dumps(item, ensure_ascii=False)
    text = get_full_text(html)
    return {
        "time": get_time(html),
        "price": get_price(html),
        "airline": get_airline(text),
        "text": text,
    }

def process_ticket_data(directory_path):
    if not os.path.exists(directory_path):
        raise FileNotFoundError(f"The path '{directory_path}' does not exist.")

    results = []

    for filename in os.listdir(directory_path):
        if filename.endswith(".json"):
            file_path = os.path.join(directory_path, filename)
            try:
                with open(file_path, "r", encoding="utf-8") as f:
                    data = json.load(f)

                items = data if isinstance(data, list) else data.values() if isinstance(data, dict) else [data]
                for item in items:
                    extracted = extract_ticket_item(item)
                    print(json.dumps(extracted, ensure_ascii=False, indent=2))
                    results.append(extracted)

                print(f"Processed {filename}: Found {len(items)} items.")

            except json.JSONDecodeError:
                print(f"Error: {filename} is not a valid JSON file.")
            except Exception as e:
                print(f"An error occurred while reading {filename}: {e}")

    return results

results = process_ticket_data("/kaggle/input/datasets/rezapourmoridi/ticket-itmes")

{
  "time": "20:15",
  "price": 9364500,
  "airline": "تابان",
  "text": "20:15 21:30 تهران (THR) مشهد (MHD) تابان تابان flight_takeoff شنبه 31 مرداد schedule 1 ساعت  و 15 دقیقه flight_takeoff شنبه 31 مرداد schedule 1 ساعت  و 15 دقیقه اکونومی 20 kg luggage airline_seat_recline_normal + 9 صندلی باقی مانده جزئیات و خرید قیمت برای هر بزرگسال یک ‌طرفه 9364500 تومان مشاهده پرواز chevron_left"
}
{
  "time": "15:50",
  "price": 9364500,
  "airline": "تابان",
  "text": "15:50 17:05 تهران (THR) مشهد (MHD) تابان تابان flight_takeoff شنبه 31 مرداد schedule 1 ساعت  و 15 دقیقه flight_takeoff شنبه 31 مرداد schedule 1 ساعت  و 15 دقیقه اکونومی 20 kg luggage airline_seat_recline_normal 1 صندلی باقی مانده جزئیات و خرید قیمت برای هر بزرگسال یک ‌طرفه 9364500 تومان مشاهده پرواز chevron_left"
}
{
  "time": "21:20",
  "price": 9400940,
  "airline": "ایران ایرتور",
  "text": "21:20 22:50 تهران (THR) مشهد (MHD) ایران ایرتور ایران ایرتور flight_takeoff شنبه 31 مرداد schedule 1 ساعت  و 30 دقیقه flight_takeoff شن